# Student At-Risk Classification – Practice Skeleton

**Short name (GitHub):** `EduRisk`  
**Lab source:** MushEdib pipeline adapted to an education / early-warning codebook.  
**Data:** `data/students.csv` (7,208 × 24 letter codes, synthetic SIS extract). Target `status`: **o** = on-track, **r** = at-risk.  
**Companion files:** `EduRisk_Solution.ipynb`, `EduRisk_Reusable_Template.ipynb`, `EduRisk.py`, `EduRisk_Cheatsheet.docx`, `EduRisk_Project_Memo.docx`, `EduRisk_Strategy_Guide.docx`, `EduRisk_1Page_Summary_Report.docx`, `edurisk_flowchart.png`.

Work top to bottom. Cells marked `# YOUR CODE HERE` are for you. Peek at the solution notebook only after you have an answer.

**This is not a grading, placement, scholarship, or expulsion engine.** A hold-out score on this codebook does not transfer to a live roster.

You will:

1. Clean `?` in `advisor` → `u`, drop 8 exact duplicates.
2. Plot status balance, attendance × status, prior-gpa × status, and a 12-feature factorize heatmap.
3. Drop zero-variance `roster-flag`, LabelEncode every column, 80/20 split (`random_state=42`).
4. Fit `RandomForestClassifier(random_state=42)` and evaluate.
5. Compare alternates, run extra practice, twist simulation knobs.



## Inline cheat-sheet (keep this cell visible)

See also **`EduRisk_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Load | `pd.read_csv("data/students.csv")` |
| Missing in this table | literal `"?"` in `advisor` (~2,073 rows) |
| Task replacement | `df["advisor"] = df["advisor"].replace("?", "u")` |
| Collision | `u` here means *unknown advisor contact*, not a grade |
| Duplicates | `df.duplicated().sum()` then `drop_duplicates()` — expect 8 |
| Zero-variance | `roster-flag` is always `e` (enrolled) — drop before encoding |
| Factorize (EDA only) | `df[col], _ = pd.factorize(df[col])` — arbitrary integer codes |
| Top-12 heatmap | drop `status` *before* `.head(12)`; heatmap is 12×12, not 13×13 |
| Encode for trees | `LabelEncoder().fit_transform` per column |
| Encode for distance / LR | `pd.get_dummies` (one-hot) |
| Split | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| RF | `RandomForestClassifier(random_state=42)` — defaults, 100 trees |
| Costly cell | actual **r**, predicted **o** (missed at-risk / skipped intervention) |
| Attendance rule | majority status per attendance code ≈ 0.71 on this table |
| Class map after LE | alphabetical → `o=0`, `r=1` |

**sklearn note.** Trees do not need scaling. LabelEncoder invents a fake order that is fine for RF / DT and wrong for kNN / unpenalized linear models. Factorize |corr| can rank a weak column above attendance — trust Gini / MI for the key feature.



## Flowchart of the desired outcome

![EduRisk flow](edurisk_flowchart.png)

Clean first (unknown advisor, drop the 8 cloned rows). Look at attendance before you fit anything. Drop `roster-flag`. Freeze the 80/20 seed at 42. Score the model on the costly cell (missed at-risk), not only accuracy. Then break it on purpose in the simulation section.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, f1_score,
)
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

try:
    import EduRisk as er
except ImportError:
    er = None
print("ready")


## 1. Data preparation

Load `data/students.csv`. Display the first 5 rows. Call `.info()`. Count `"?"` in `advisor` and replace them with `"u"`. Count and drop exact duplicate rows. Print the cleaned shape and `status` counts.

Expected: 7,208 raw rows × 24 columns, ~2,073 question marks, 8 duplicates → **7,200** cleaned rows. Status split about 3,832 on-track / 3,368 at-risk.


In [ ]:
# YOUR CODE HERE
df = None
print("First 5 rows:")
# print(df.head())
print("DataFrame info:")
# df.info()
n_q = None
print("Missing ('?') values in 'advisor':", n_q)
# replace '?' with 'u'
n_dups = None
print("Number of duplicate rows:", n_dups)
print("Cleaned dataset shape:", None)
# print(df["status"].value_counts())


### Alternate — treat `?` as `missing`, or impute the mode

The brief asks for `"u"`. Two other legal choices: a new token `"missing"`, or the mode of observed advisor codes (`b` = brief). Trees can learn from an explicit missing level; imputing the mode invents a contact pattern that was never recorded.


In [ ]:
# YOUR CODE HERE — do not overwrite df; use a copy
df_mode = None
print("mode of raw advisor (excluding ?):", None)


## 2. Exploratory data analysis

Plot three countplots, then a temporary factorize encoding, then the **12 × 12** heatmap of the top features by |corr| with `status`.

1. Status balance.
2. Attendance vs status (the key feature).
3. Prior-GPA vs status (overlap — weaker on its own).
4. `pd.factorize` every column into integers.
5. Absolute correlation with `status`, drop the target, take `.head(12)`, heatmap **those 12 only**.

Reference images: `edurisk_class_balance.png`, `edurisk_attendance.png`, `edurisk_priorgpa.png`, `edurisk_heatmap.png`.

**Caveat.** Factorize assigns arbitrary integers, so |corr| can put `homework` above `attendance` even when Gini / MI agree attendance is the driver. Treat the heatmap as a sketch.


In [ ]:
# YOUR CODE HERE
# 1. status countplot
# 2. attendance vs status
# 3. prior-gpa vs status
# 4. df_encoded = factorize copy
# 5. correlations / top12 / heatmap of exactly 12 columns
top12 = None
print(top12)


### What you should see

- Balance is close: **~53% on-track / ~47% at-risk**. Accuracy is usable; the *costly* error is still a missed at-risk flag.
- Attendance nearly partitions the tails: `a` (always) is mostly on-track; `r` / `n` (rarely / never) are mostly at-risk. `s` and `u` overlap — that is where homework, GPA, and tardies earn their keep.
- Prior GPA overlaps both classes. High GPA helps but does not guarantee on-track.
- Expect factorize |corr| to mention homework, prior-gpa, attendance, extra-help. Confirm with Gini / MI later.


## 3. Preprocessing

Drop `roster-flag`. LabelEncode **every remaining column including `status`**. Split `X` / `y`. `train_test_split(..., test_size=0.2, random_state=42)`. Print the four shapes.

Expected shapes: `X_train (5760, 22)`, `X_test (1440, 22)`.


In [ ]:
# YOUR CODE HERE
X = None
y = None
X_train = X_test = y_train = y_test = None
print("X_train shape:", None)


### Alternate — one-hot instead of LabelEncoder

Keep this for logistic regression later. Do not replace `X_train` used by the forest.


In [ ]:
# YOUR CODE HERE
X_oh = None
print("one-hot width:", None)


## 4. Random Forest

Initialize `RandomForestClassifier(random_state=42)`, fit on the training fold, predict `X_test` into `y_pred`. Name the model `clf` so the evaluation cell matches the brief.


In [ ]:
# YOUR CODE HERE
clf = None
y_pred = None


## 5. Model evaluation

Print accuracy, the numeric confusion matrix, and the classification report. Heatmap the matrix. Horizontal bar of the **top 5** Gini importances.

On this seed the default forest sits near **0.76** accuracy. Watch the costly cell: actual at-risk, predicted on-track (missed intervention).


In [ ]:
# YOUR CODE HERE
accuracy = None
cm = None
print("Accuracy score:", accuracy)
print("Confusion matrix:\n", cm)
# heatmap
# top-5 importance bar
importances = None


## 6. Alternate code that reaches a similar decision

Fit a single `DecisionTreeClassifier`, a one-hot `LogisticRegression`, an attendance-majority rule, and a mutual-information ranking. On this table the one-hot logistic model is often a hair *above* the forest; the attendance rule sits near **0.71**.


In [ ]:
# YOUR CODE HERE
dt = None
lr = None
att_acc = None
mi = None
print("DT acc", None)
print("LR acc", None)
print("attendance-rule acc", att_acc)
print(mi)


## 7. More practice

**A.** Restrict the test fold to `setting == r` (rural). Does accuracy hold?

**B.** Cost matrix. Treat a missed at-risk flag as 5× worse than a false alarm. Sweep `predict_proba` thresholds that flag at-risk more aggressively. How many extra false alarms buy a drop in missed-risk cells?

**C.** A 2-feature card: `attendance` + `homework` only. Compare test accuracy and the costly-cell count to the full 22-feature forest.


In [ ]:
# YOUR CODE HERE
print("practice A/B/C")


## 8. Simulation — twist a few knobs

Default RF is *not* perfect here (unlike MushEdib). Knobs that move:

| Knob | What we change | What usually happens on this table |
|------|----------------|------------------------------------|
| `max_depth` | 1 → None | depth 1 ≈ 0.65; depth 8 ≈ 0.75 |
| drop features | remove attendance / homework | drops toward 0.61–0.65; attendance-only ≈ 0.71 |
| label flip | flip 0–35% of *train* labels | holds near 0.75 until ~10%, then falls |
| training n | 50, 100, …, 5,760 | 50 rows ≈ 0.59; 800 rows already ~0.75 |

Edit the lists, re-run, read the 2×2 panel. Reference: `edurisk_simulation.png`.


In [ ]:
# YOUR CODE HERE — change these lists
DEPTHS = [1, 2, 3, 4, 5, 6, 8, None]
FLIP_RATES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
TRAIN_NS = [50, 100, 200, 400, 800, 1600, 3200, len(X_train)]


## 9. Audience notes (rewrite the same result four ways)

Use the attached audience PDFs. Same numbers, four pitches.

| Audience | Data literacy | Subject knowledge | Time span | What to show |
|----------|---------------|-------------------|-----------|--------------|
| Expert (education researcher) | high | high | long | MI vs Gini vs factorize, why 0.76 is not a placement rule, `u` vs mode impute |
| Technician (SIS / early-warning) | medium | high practical | short | 2-feature card, threshold as intervention capacity, do not write to the gradebook |
| Executive (dean / superintendent) | low–medium | low–medium | very short | 53 / 47 balance, ~0.76 acc, 183 missed-risk on hold-out, human review required |
| Nonspecialist (family) | low | low | short | “showing up is the loud clue in *this* table, not a verdict about your child” |

Full prose: `EduRisk_Project_Memo.docx`.



## 10. Good fit vs limitations

**Good fit**
- All-categorical SIS / early-warning codebook with a nearly balanced binary target.
- Tree ensembles and a one-hot logistic baseline.
- Teaching clean vs mode-impute, LabelEncoder vs one-hot, costly-error thinking (missed intervention).

**Limitations / anti-applications**
- Synthetic extract. New schools, different coding manuals, and mid-term grade streams are out of scope.
- `?` → `u` is a teaching token, not a district standard.
- ~0.76 hold-out accuracy is *not* a reason to auto-assign tutoring, hold a student back, or deny a scholarship.
- Never a grading engine, never an expulsion score, never a public “will my child fail?” app.

Top applications of the *pattern* (categorical RF + costly FN): attendance early-warning, library-fines typology, course-add codes — always with a counselor in the loop.

